# Adversarial Frozen-Opponent MAPPO

Small launcher for the experimental two-team JAX MAPPO lane. It runs the adversarial workflow config, optionally actor-warm-starting from a compatible cooperative checkpoint.

In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training cells so JAX sees the memory settings.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PACKAGE_ROOT = PROJECT_ROOT / "src" / "ant_byte_env"
sys.path.insert(0, str(PACKAGE_ROOT.parent))

CONFIG = PROJECT_ROOT / "experiments" / "adversarial_frozen_opponent_shared_writes_8ants_strided_cnn_anchor0075.json"
SOURCE_EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "exploration_to_forage_full_layout_8ants_half_food_50x50_shared_writes.json"
RUN_ROOT = PROJECT_ROOT / "runs" / "notebooks" / "adversarial_frozen_opponent"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(CONFIG)
print(SOURCE_EXPERIMENT_CONFIG)
print(RUN_ROOT)

In [ ]:
import importlib
import json
import shutil

import jax
from IPython.display import Video, display
from tqdm.auto import tqdm

from ant_byte_env import AntByteForagingEnv, notebook_workflows as workflows
from ant_byte_env.experiments import config_args_to_argv, load_experiment_config
from ant_byte_env.training.jax_mappo import updates as jax_mappo_updates
from ant_byte_env.training.jax_mappo.adversarial import cli as adversarial_cli
from ant_byte_env.training.jax_mappo.adversarial.checkpointing import evaluate_checkpoint_matrix
from ant_byte_env.training.jax_mappo.adversarial.rendering import render_adversarial_rollout
from ant_byte_env.training.jax_mappo.adversarial import runner as adversarial_runner

workflows = importlib.reload(workflows)
jax_mappo_updates = importlib.reload(jax_mappo_updates)
adversarial_cli = importlib.reload(adversarial_cli)
adversarial_runner = importlib.reload(adversarial_runner)

resource_status = workflows.configure_jax_notebook_runtime(memory_fraction="0.35")
workflows.assert_notebook_resources_available(resource_status)
print(f"JAX device: {jax.devices()[0]}")
resource_status


## Training Settings

In [ ]:
RUN_TRAINING = True
EVALUATION_EPISODES = 32
EVALUATION_MAX_STEPS = 2000
EVALUATION_PROGRESS_INTERVAL = 100
RENDER_SECONDS = 60
RENDER_FPS = AntByteForagingEnv.metadata["render_fps"]
RENDER_MAX_FRAMES = RENDER_SECONDS * RENDER_FPS
FIXED_RENDER_HUB_POSITIONS = [[21, 25], [29, 25]]
FIXED_RENDER_FOOD_POSITIONS = [[25, 23], [25, 27]]
FIXED_RENDER_FOOD_COUNT = 125
RUN_NAME = "cnn_critic_strided_warmstart_good_policy_eval32_lr5e5_anchor0075_notebook"
CURRICULUM_ROOT = RUN_ROOT / RUN_NAME
CURRICULUM_ROOT.mkdir(parents=True, exist_ok=True)
FIXED_RENDER_VIDEO_PATH = CURRICULUM_ROOT / "media" / "adversarial_rollout_fixed_center.mp4"

# Start the CNN-critic run actor-only from the useful MLP-critic adversarial policy.
# Do not resume the old checkpoint wholesale, because its critic is an MLP.
CURRICULUM_START_CHECKPOINT = None

# Controlled diagnostic layout: hubs are sampled from the interior, with food near their midpoint.
DIAGNOSTIC_PLACEMENT_OVERRIDES = [
    "--food-count", "125",
    "--food-sources", "4",
    "--layout-margin", "6",
    "--hub-center-window-size", "0",
    "--hub-pair-distance-min", "12",
    "--hub-pair-distance-max", "20",
    "--food-midpoint-window-size", "8",
]

# Fixed-N CNN critic continuation, trained in two notebook-sized chunks.
# The first chunk uses learner-load-model for actor-only transfer; later chunks resume CNN checkpoints.
# No reward shaping is added here; training reward stays own deliveries minus opponent deliveries.
CURRICULUM_UPDATES_PER_STAGE = 500
CRITIC_WARMUP_UPDATES = 0
TRAINING_CHUNK_UPDATES = 250
BEHAVIOR_ANCHOR_COEF = 0.0075
BEST_CHECKPOINT_SELECTION = "eval"
BEST_CHECKPOINT_METRIC = "eval_learner_vs_frozen_side_swap_adjusted_delivery_difference"
BEST_CHECKPOINT_MODE = "max"
BEST_EVAL_EPISODES = 32
BEST_EVAL_INTERVAL = TRAINING_CHUNK_UPDATES
CRITIC_WARMUP_BEST_METRIC = "value_loss"
CRITIC_WARMUP_BEST_MODE = "min"
FOOD_CURRICULUM = [
    {
        "label": "target_125food_2src_dist20_36_mid16_eval32_lr5e5_anchor0075_team0_strided_cnn",
        "food_count": 125,
        "food_sources": 2,
        "layout_margin": 6,
        "hub_center_window_size": 0,
        "hub_pair_distance_min": 20,
        "hub_pair_distance_max": 36,
        "food_midpoint_window_size": 16,
        "learner_team": 0,
        "ent_coef": 0.00015,
        "learning_rate": 5e-5,
        "behavior_anchor_coef": BEHAVIOR_ANCHOR_COEF,
    },
]

STAGE_ARG_KEYS = {
    "food_count": "food-count",
    "food_sources": "food-sources",
    "layout_margin": "layout-margin",
    "hub_center_window_size": "hub-center-window-size",
    "hub_pair_distance_min": "hub-pair-distance-min",
    "hub_pair_distance_max": "hub-pair-distance-max",
    "food_midpoint_window_size": "food-midpoint-window-size",
    "learner_team": "learner-team",
    "ent_coef": "ent-coef",
    "learning_rate": "learning-rate",
}

spec = load_experiment_config(CONFIG)
source_spec = load_experiment_config(SOURCE_EXPERIMENT_CONFIG)
LEARNER_WARMSTART_CHECKPOINT = (PROJECT_ROOT / spec.args["learner_load_model"]).resolve()
BEHAVIOR_ANCHOR_CHECKPOINT = (PROJECT_ROOT / spec.args["behavior_anchor_model"]).resolve()
FROZEN_OPPONENT_CHECKPOINT = (PROJECT_ROOT / spec.args["opponent_load_model"]).resolve()
for checkpoint_path in (
    LEARNER_WARMSTART_CHECKPOINT,
    BEHAVIOR_ANCHOR_CHECKPOINT,
    FROZEN_OPPONENT_CHECKPOINT,
):
    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)
if CURRICULUM_START_CHECKPOINT is not None and not CURRICULUM_START_CHECKPOINT.exists():
    raise FileNotFoundError(CURRICULUM_START_CHECKPOINT)

BASE_TRAINING_ARGS = dict(spec.args)
BASE_TRAINING_ARGS.pop("learner_load_model", None)
BASE_TRAINING_ARGS.pop("run_dir", None)
BASE_TRAINING_ARGS.pop("save_best_model", None)
BASE_TRAINING_ARGS.pop("total_timesteps", None)
WANDB_RUN_NAME_PREFIX = BASE_TRAINING_ARGS.pop("wandb_run_name", spec.name)
BASE_TRAINING_ARGV = config_args_to_argv(BASE_TRAINING_ARGS)
steps_per_update = int(spec.args["num_envs"]) * int(spec.args["num_steps"])
stage_timesteps = int(CURRICULUM_UPDATES_PER_STAGE * steps_per_update)
critic_warmup_timesteps = int(CRITIC_WARMUP_UPDATES * steps_per_update)
training_chunk_timesteps = int(TRAINING_CHUNK_UPDATES * steps_per_update)
CURRICULUM_SUMMARY_PATH = CURRICULUM_ROOT / "curriculum_summary.json"


def stage_run_dir(stage_index):
    stage = FOOD_CURRICULUM[stage_index]
    return CURRICULUM_ROOT / f"stage_{stage_index + 1:02d}_{stage['label']}"


def stage_checkpoint(stage_index):
    return stage_run_dir(stage_index) / "checkpoints" / "best_model.pkl"


def stage_summary_path(stage_index):
    return stage_run_dir(stage_index) / "stage_summary.json"


def chunk_plan(stage_index):
    del stage_index
    chunks = []
    next_index = 0
    offset = 0
    if CRITIC_WARMUP_UPDATES > 0:
        chunks.append({
            "chunk_index": next_index,
            "label": "critic_warmup",
            "start_update": offset,
            "updates": CRITIC_WARMUP_UPDATES,
            "freeze_actor": True,
        })
        next_index += 1
        offset += CRITIC_WARMUP_UPDATES
    remaining_updates = max(0, CURRICULUM_UPDATES_PER_STAGE - offset)
    train_index = 1
    while remaining_updates > 0:
        chunk_updates = min(TRAINING_CHUNK_UPDATES, remaining_updates)
        chunks.append({
            "chunk_index": next_index,
            "label": f"train_{train_index:02d}",
            "start_update": offset,
            "updates": chunk_updates,
            "freeze_actor": False,
        })
        next_index += 1
        train_index += 1
        offset += chunk_updates
        remaining_updates -= chunk_updates
    return chunks


def chunk_run_dir(stage_index, chunk):
    return stage_run_dir(stage_index) / f"chunk_{chunk['chunk_index']:02d}_{chunk['label']}"


def chunk_checkpoint(stage_index, chunk):
    return chunk_run_dir(stage_index, chunk) / "checkpoints" / "model.pkl"


def chunk_best_checkpoint(stage_index, chunk):
    return chunk_run_dir(stage_index, chunk) / "checkpoints" / "best_model.pkl"


def build_stage_overrides(
    stage_index,
    resume_checkpoint=None,
    *,
    run_dir=None,
    update_count=None,
    freeze_actor=False,
):
    stage = FOOD_CURRICULUM[stage_index]
    update_count = CURRICULUM_UPDATES_PER_STAGE if update_count is None else int(update_count)
    run_dir = stage_run_dir(stage_index) if run_dir is None else run_dir
    best_model_path = Path(run_dir) / "checkpoints" / "best_model.pkl"
    if freeze_actor:
        best_selection = "train"
        best_metric = CRITIC_WARMUP_BEST_METRIC
        best_mode = CRITIC_WARMUP_BEST_MODE
        behavior_anchor_coef = 0.0
    else:
        best_selection = BEST_CHECKPOINT_SELECTION
        best_metric = BEST_CHECKPOINT_METRIC
        best_mode = BEST_CHECKPOINT_MODE
        behavior_anchor_coef = float(stage.get("behavior_anchor_coef", BEHAVIOR_ANCHOR_COEF))
    overrides = [
        "--run-dir",
        str(run_dir),
        "--total-timesteps",
        str(int(update_count) * steps_per_update),
        "--eval-episodes",
        "0",
        "--save-best-model",
        str(best_model_path),
        "--best-model-selection",
        best_selection,
        "--best-model-metric",
        best_metric,
        "--best-model-mode",
        best_mode,
        "--best-eval-episodes",
        str(BEST_EVAL_EPISODES),
        "--best-eval-interval",
        str(BEST_EVAL_INTERVAL),
        "--behavior-anchor-coef",
        str(behavior_anchor_coef),
    ]
    for key, option_name in STAGE_ARG_KEYS.items():
        overrides.extend([f"--{option_name}", str(stage[key])])
    if freeze_actor:
        overrides.append("--freeze-actor")
    overrides.extend([
        "--wandb-run-name",
        f"{WANDB_RUN_NAME_PREFIX}-{Path(run_dir).name}",
    ])
    if resume_checkpoint is None:
        overrides.extend(["--learner-load-model", str(LEARNER_WARMSTART_CHECKPOINT)])
    else:
        overrides.extend([
            "--resume-model",
            str(resume_checkpoint),
            "--opponent-load-model",
            str(FROZEN_OPPONENT_CHECKPOINT),
        ])
    return overrides


def build_stage_argv(stage_index, resume_checkpoint=None, **kwargs):
    return [*BASE_TRAINING_ARGV, *build_stage_overrides(stage_index, resume_checkpoint, **kwargs)]


def build_diagnostic_argv(argv):
    return [*argv, *DIAGNOSTIC_PLACEMENT_OVERRIDES]


def build_fixed_scene_argv(argv):
    return [
        *argv,
        "--food-count", str(FIXED_RENDER_FOOD_COUNT),
        "--food-sources", str(len(FIXED_RENDER_FOOD_POSITIONS)),
    ]


FINAL_STAGE_INDEX = len(FOOD_CURRICULUM) - 1
CHECKPOINT_PATH = stage_checkpoint(FINAL_STAGE_INDEX)
preview_chunk = chunk_plan(0)[0]
training_argv = build_stage_argv(
    0,
    CURRICULUM_START_CHECKPOINT,
    run_dir=chunk_run_dir(0, preview_chunk),
    update_count=preview_chunk["updates"],
    freeze_actor=preview_chunk["freeze_actor"],
)
diagnostic_argv = build_diagnostic_argv(training_argv)
fixed_scene_argv = build_fixed_scene_argv(training_argv)
ACTIVE_CHECKPOINT_PATH = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else None

json.dumps({
    "experiment": spec.name,
    "source_experiment": source_spec.name,
    "learner_warmstart_checkpoint": str(LEARNER_WARMSTART_CHECKPOINT),
    "behavior_anchor_checkpoint": str(BEHAVIOR_ANCHOR_CHECKPOINT),
    "frozen_opponent_checkpoint": str(FROZEN_OPPONENT_CHECKPOINT),
    "curriculum_start_checkpoint": str(CURRICULUM_START_CHECKPOINT) if CURRICULUM_START_CHECKPOINT else None,
    "critic_architecture": spec.args["critic_architecture"],
    "curriculum_root": str(CURRICULUM_ROOT),
    "updates_per_stage": CURRICULUM_UPDATES_PER_STAGE,
    "critic_warmup_updates": CRITIC_WARMUP_UPDATES,
    "training_chunk_updates": TRAINING_CHUNK_UPDATES,
    "best_checkpoint_selection": BEST_CHECKPOINT_SELECTION,
    "best_checkpoint_metric": BEST_CHECKPOINT_METRIC,
    "best_checkpoint_mode": BEST_CHECKPOINT_MODE,
    "best_eval_episodes": BEST_EVAL_EPISODES,
    "best_eval_interval": BEST_EVAL_INTERVAL,
    "behavior_anchor_coef": BEHAVIOR_ANCHOR_COEF,
    "stage_timesteps": stage_timesteps,
    "critic_warmup_timesteps": critic_warmup_timesteps,
    "training_chunk_timesteps": training_chunk_timesteps,
    "total_curriculum_updates": CURRICULUM_UPDATES_PER_STAGE * len(FOOD_CURRICULUM),
    "total_curriculum_timesteps": stage_timesteps * len(FOOD_CURRICULUM),
    "render_seconds": RENDER_SECONDS,
    "render_max_frames": RENDER_MAX_FRAMES,
    "evaluation_episodes": EVALUATION_EPISODES,
    "evaluation_max_steps": EVALUATION_MAX_STEPS,
    "fixed_render_hub_positions": FIXED_RENDER_HUB_POSITIONS,
    "fixed_render_food_positions": FIXED_RENDER_FOOD_POSITIONS,
    "fixed_render_food_count": FIXED_RENDER_FOOD_COUNT,
    "fixed_render_video_path": str(FIXED_RENDER_VIDEO_PATH),
    "fixed_eval_artifact": str(CURRICULUM_ROOT / f"fixed_center_eval_{EVALUATION_EPISODES}ep_{EVALUATION_MAX_STEPS}step.json"),
    "diagnostic_placement_overrides": DIAGNOSTIC_PLACEMENT_OVERRIDES,
    "final_checkpoint": str(CHECKPOINT_PATH),
    "active_checkpoint": str(ACTIVE_CHECKPOINT_PATH) if ACTIVE_CHECKPOINT_PATH else None,
    "chunk_plan_preview": chunk_plan(0),
    "food_curriculum": FOOD_CURRICULUM,
}, indent=2)


In [ ]:
from ant_byte_env.cli import main as ant_byte_main

preview_chunk = chunk_plan(0)[0]
ant_byte_main([
    "train",
    "jax",
    "--config",
    str(CONFIG),
    "--dry-run",
    "--",
    *build_stage_overrides(
        0,
        CURRICULUM_START_CHECKPOINT,
        run_dir=chunk_run_dir(0, preview_chunk),
        update_count=preview_chunk["updates"],
        freeze_actor=preview_chunk["freeze_actor"],
    ),
])


## Run

In [ ]:
total_curriculum_updates = CURRICULUM_UPDATES_PER_STAGE * len(FOOD_CURRICULUM)
progress_rows = []
stage_results = []
progress_state = {"completed_updates": 0}


def checkpoint_score(metrics):
    return float(metrics.get(BEST_CHECKPOINT_METRIC, float("-inf")))


def make_stage_progress_callback(stage_index, stage, chunk):
    def record_progress(update_index, total_update_count, train_metrics):
        completed_updates = (
            stage_index * CURRICULUM_UPDATES_PER_STAGE
            + int(chunk["start_update"])
            + int(update_index)
        )
        update_delta = max(0, completed_updates - progress_state["completed_updates"])
        progress_state["completed_updates"] = completed_updates
        if progress_bar is not None and update_delta:
            progress_bar.update(update_delta)
            progress_bar.set_postfix(
                stage=f"{stage_index + 1}/{len(FOOD_CURRICULUM)}",
                chunk=chunk["label"],
                freeze="Y" if chunk["freeze_actor"] else "N",
                food=f"{stage['food_count']}/{stage['food_sources']}",
                hub=f"{stage['hub_center_window_size']}",
                margin=f"{stage['layout_margin']}",
                dist=f"{stage['hub_pair_distance_min']}-{stage['hub_pair_distance_max']}",
                ent=f"{stage['ent_coef']:.4f}",
                loss=f"{train_metrics['loss']:.3f}",
                ret=f"{train_metrics['episode_return']:.2f}",
                diff=f"{train_metrics['delivery_event_difference']:.0f}",
                anchor=f"{train_metrics.get('behavior_anchor_kl', 0.0):.3f}",
            )
        progress_rows.append({
            "stage_index": int(stage_index + 1),
            "stage_label": stage["label"],
            "chunk": dict(chunk),
            "update": int(update_index),
            "total_updates": int(total_update_count),
            **train_metrics,
        })

    return record_progress


if RUN_TRAINING:
    previous_checkpoint = CURRICULUM_START_CHECKPOINT
    progress_bar = tqdm(
        total=total_curriculum_updates,
        desc="adversarial warmup/chunk curriculum",
        bar_format="{desc}: {n_fmt}/{total_fmt} updates |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )
    try:
        for stage_index, stage in enumerate(FOOD_CURRICULUM):
            current_resume_checkpoint = previous_checkpoint
            chunk_results = []
            best_chunk = None
            for chunk in chunk_plan(stage_index):
                current_run_dir = chunk_run_dir(stage_index, chunk)
                current_argv = build_stage_argv(
                    stage_index,
                    current_resume_checkpoint,
                    run_dir=current_run_dir,
                    update_count=chunk["updates"],
                    freeze_actor=chunk["freeze_actor"],
                )
                metrics = adversarial_runner.main(
                    current_argv,
                    progress_callback=make_stage_progress_callback(stage_index, stage, chunk),
                )
                chunk_summary_path = current_run_dir / "summary.json"
                chunk_summary = (
                    json.loads(chunk_summary_path.read_text(encoding="utf-8"))
                    if chunk_summary_path.exists()
                    else {}
                )
                runner_best_checkpoint = Path(
                    chunk_summary.get("best_checkpoint_path") or chunk_best_checkpoint(stage_index, chunk)
                )
                final_checkpoint = chunk_checkpoint(stage_index, chunk)
                current_checkpoint = runner_best_checkpoint if runner_best_checkpoint.exists() else final_checkpoint
                if not current_checkpoint.exists():
                    raise FileNotFoundError(current_checkpoint)
                checkpoint_metrics = chunk_summary.get("best_checkpoint_metrics") or metrics
                eligible_for_stage_best = (
                    not chunk["freeze_actor"] and BEST_CHECKPOINT_METRIC in checkpoint_metrics
                )
                chunk_result = {
                    "stage_index": stage_index + 1,
                    "stage_label": stage["label"],
                    "chunk": dict(chunk),
                    "checkpoint": str(current_checkpoint),
                    "final_checkpoint": str(final_checkpoint),
                    "runner_best_checkpoint": str(runner_best_checkpoint),
                    "resume_checkpoint": str(current_resume_checkpoint) if current_resume_checkpoint else None,
                    "metrics": checkpoint_metrics,
                    "final_metrics": metrics,
                    "score": checkpoint_score(checkpoint_metrics),
                    "eligible_for_stage_best": eligible_for_stage_best,
                    "runner_summary": chunk_summary,
                    "argv": current_argv,
                }
                chunk_results.append(chunk_result)
                if eligible_for_stage_best and (
                    best_chunk is None or chunk_result["score"] > best_chunk["score"]
                ):
                    best_chunk = chunk_result
                current_resume_checkpoint = current_checkpoint

            if best_chunk is None:
                best_chunk = max(chunk_results, key=lambda row: row["score"])

            best_checkpoint = stage_checkpoint(stage_index)
            best_checkpoint.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(best_chunk["checkpoint"], best_checkpoint)
            stage_result = {
                "stage_index": stage_index + 1,
                "stage": stage,
                "best_checkpoint": str(best_checkpoint),
                "best_source_checkpoint": best_chunk["checkpoint"],
                "best_chunk": best_chunk["chunk"],
                "best_score_metric": BEST_CHECKPOINT_METRIC,
                "best_score": best_chunk["score"],
                "best_metrics": best_chunk["metrics"],
                "best_argv": best_chunk["argv"],
                "chunks": chunk_results,
            }
            stage_summary_path(stage_index).write_text(
                json.dumps(stage_result, indent=2),
                encoding="utf-8",
            )
            stage_results.append(stage_result)
            previous_checkpoint = best_checkpoint
    finally:
        progress_bar.close()

    training_argv = stage_results[-1]["best_argv"]
    diagnostic_argv = build_diagnostic_argv(training_argv)
    fixed_scene_argv = build_fixed_scene_argv(training_argv)
    metrics = stage_results[-1]["best_metrics"]
    ACTIVE_CHECKPOINT_PATH = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else None
    CURRICULUM_SUMMARY_PATH.write_text(
        json.dumps({
            "curriculum_root": str(CURRICULUM_ROOT),
            "learner_warmstart_checkpoint": str(LEARNER_WARMSTART_CHECKPOINT),
            "behavior_anchor_checkpoint": str(BEHAVIOR_ANCHOR_CHECKPOINT),
            "frozen_opponent_checkpoint": str(FROZEN_OPPONENT_CHECKPOINT),
            "curriculum_start_checkpoint": str(CURRICULUM_START_CHECKPOINT) if CURRICULUM_START_CHECKPOINT else None,
            "critic_architecture": spec.args["critic_architecture"],
            "updates_per_stage": CURRICULUM_UPDATES_PER_STAGE,
            "critic_warmup_updates": CRITIC_WARMUP_UPDATES,
            "training_chunk_updates": TRAINING_CHUNK_UPDATES,
            "best_checkpoint_selection": BEST_CHECKPOINT_SELECTION,
            "best_checkpoint_metric": BEST_CHECKPOINT_METRIC,
            "best_checkpoint_mode": BEST_CHECKPOINT_MODE,
            "best_eval_episodes": BEST_EVAL_EPISODES,
            "best_eval_interval": BEST_EVAL_INTERVAL,
            "behavior_anchor_coef": BEHAVIOR_ANCHOR_COEF,
            "stage_timesteps": stage_timesteps,
            "total_curriculum_updates": total_curriculum_updates,
            "total_curriculum_timesteps": stage_timesteps * len(FOOD_CURRICULUM),
            "final_checkpoint": str(CHECKPOINT_PATH),
            "diagnostic_placement_overrides": DIAGNOSTIC_PLACEMENT_OVERRIDES,
            "fixed_render_hub_positions": FIXED_RENDER_HUB_POSITIONS,
            "fixed_render_food_positions": FIXED_RENDER_FOOD_POSITIONS,
            "fixed_render_food_count": FIXED_RENDER_FOOD_COUNT,
            "stages": stage_results,
        }, indent=2),
        encoding="utf-8",
    )
else:
    progress_bar = None
    metrics = {"status": "skipped", "set_RUN_TRAINING": True}

{"metrics": metrics, "stage_results": stage_results, "progress_rows": progress_rows[-5:]}


In [ ]:
if CURRICULUM_SUMMARY_PATH.exists():
    curriculum_summary = json.loads(CURRICULUM_SUMMARY_PATH.read_text(encoding="utf-8"))
    compact_summary = {
        "final_checkpoint": curriculum_summary["final_checkpoint"],
        "updates_per_stage": curriculum_summary["updates_per_stage"],
        "critic_warmup_updates": curriculum_summary.get("critic_warmup_updates"),
        "training_chunk_updates": curriculum_summary.get("training_chunk_updates"),
        "best_checkpoint_metric": curriculum_summary.get("best_checkpoint_metric"),
        "stages": [
            {
                "stage_index": stage["stage_index"],
                "stage": stage["stage"],
                "best_checkpoint": stage["best_checkpoint"],
                "best_chunk": stage["best_chunk"],
                "best_score": stage["best_score"],
                "best_metrics": stage["best_metrics"],
            }
            for stage in curriculum_summary["stages"]
        ],
    }
    print(json.dumps(compact_summary, indent=2, sort_keys=True))
else:
    print(f"No curriculum summary yet: {CURRICULUM_SUMMARY_PATH}")

for stage_index, stage in enumerate(FOOD_CURRICULUM):
    summary_path = stage_summary_path(stage_index)
    if summary_path.exists():
        print(summary_path)
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        print(json.dumps({
            "best_checkpoint": summary["best_checkpoint"],
            "best_chunk": summary["best_chunk"],
            "best_score": summary["best_score"],
        }, indent=2, sort_keys=True))
    for metrics_path in sorted(stage_run_dir(stage_index).glob("chunk_*/metrics.jsonl"))[-3:]:
        print(metrics_path)
        print(metrics_path.read_text(encoding="utf-8").splitlines()[-1])


## Evaluation

In [ ]:
if ACTIVE_CHECKPOINT_PATH is not None and ACTIVE_CHECKPOINT_PATH.exists():
    matchup_order = [
        "frozen_vs_frozen",
        "learner_vs_frozen",
        "frozen_vs_learner",
        "random_vs_frozen",
        "learner_vs_random",
    ]
    matchup_offsets = {name: index for index, name in enumerate(matchup_order)}
    evaluation_total_steps = len(matchup_order) * EVALUATION_EPISODES * EVALUATION_MAX_STEPS
    evaluation_progress_rows = []
    evaluation_progress_state = {"completed_steps": 0}
    evaluation_bar = tqdm(
        total=evaluation_total_steps,
        desc="adversarial robust eval",
        bar_format="{desc}: {n_fmt}/{total_fmt} steps |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )

    def record_evaluation_progress(matchup_name, episode_index, total_episodes, eval_metrics_row):
        matchup_offset = matchup_offsets[matchup_name]
        completed_steps = int(
            (
                matchup_offset * total_episodes * EVALUATION_MAX_STEPS
                + (episode_index - 1) * EVALUATION_MAX_STEPS
                + eval_metrics_row["step"]
            )
        )
        step_delta = max(0, completed_steps - evaluation_progress_state["completed_steps"])
        evaluation_progress_state["completed_steps"] = completed_steps
        if step_delta:
            evaluation_bar.update(step_delta)
        evaluation_bar.set_postfix(
            matchup=matchup_name,
            episode=f"{episode_index}/{total_episodes}",
            step=f"{int(eval_metrics_row['step'])}/{EVALUATION_MAX_STEPS}",
            event=eval_metrics_row["event"],
        )
        evaluation_progress_rows.append({
            "matchup": matchup_name,
            "episode": int(episode_index),
            "total_episodes": int(total_episodes),
            **eval_metrics_row,
        })

    try:
        evaluation_metrics = evaluate_checkpoint_matrix(
            ACTIVE_CHECKPOINT_PATH,
            argv=training_argv,
            eval_episodes=EVALUATION_EPISODES,
            eval_max_steps=EVALUATION_MAX_STEPS,
            progress_callback=record_evaluation_progress,
            progress_step_interval=EVALUATION_PROGRESS_INTERVAL,
        )
    finally:
        evaluation_bar.close()
    evaluation_artifact = CURRICULUM_ROOT / f"final_eval_{EVALUATION_EPISODES}ep_{EVALUATION_MAX_STEPS}step.json"
    evaluation_artifact.write_text(json.dumps(evaluation_metrics, indent=2, sort_keys=True), encoding="utf-8")
else:
    evaluation_progress_rows = []
    evaluation_metrics = {"status": "missing_checkpoint", "checkpoint": str(ACTIVE_CHECKPOINT_PATH) if ACTIVE_CHECKPOINT_PATH else None}
    evaluation_artifact = None

{
    "evaluation_artifact": str(evaluation_artifact) if evaluation_artifact else None,
    "evaluation_metrics": evaluation_metrics,
    "evaluation_progress_rows": evaluation_progress_rows,
}


## Fixed-Scene Evaluation

In [ ]:
if ACTIVE_CHECKPOINT_PATH is not None and ACTIVE_CHECKPOINT_PATH.exists():
    fixed_matchup_order = [
        "frozen_vs_frozen",
        "learner_vs_frozen",
        "frozen_vs_learner",
        "random_vs_frozen",
        "learner_vs_random",
    ]
    fixed_matchup_offsets = {name: index for index, name in enumerate(fixed_matchup_order)}
    fixed_evaluation_total_steps = len(fixed_matchup_order) * EVALUATION_EPISODES * EVALUATION_MAX_STEPS
    fixed_evaluation_progress_rows = []
    fixed_evaluation_progress_state = {"completed_steps": 0}
    fixed_evaluation_bar = tqdm(
        total=fixed_evaluation_total_steps,
        desc="fixed-scene eval",
        bar_format="{desc}: {n_fmt}/{total_fmt} steps |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )

    def record_fixed_evaluation_progress(matchup_name, episode_index, total_episodes, eval_metrics_row):
        matchup_offset = fixed_matchup_offsets[matchup_name]
        completed_steps = int(
            (
                matchup_offset * total_episodes * EVALUATION_MAX_STEPS
                + (episode_index - 1) * EVALUATION_MAX_STEPS
                + eval_metrics_row["step"]
            )
        )
        step_delta = max(0, completed_steps - fixed_evaluation_progress_state["completed_steps"])
        fixed_evaluation_progress_state["completed_steps"] = completed_steps
        if step_delta:
            fixed_evaluation_bar.update(step_delta)
        fixed_evaluation_bar.set_postfix(
            matchup=matchup_name,
            episode=f"{episode_index}/{total_episodes}",
            step=f"{int(eval_metrics_row['step'])}/{EVALUATION_MAX_STEPS}",
            event=eval_metrics_row["event"],
        )
        fixed_evaluation_progress_rows.append({
            "matchup": matchup_name,
            "episode": int(episode_index),
            "total_episodes": int(total_episodes),
            **eval_metrics_row,
        })

    try:
        raw_fixed_evaluation_metrics = evaluate_checkpoint_matrix(
            ACTIVE_CHECKPOINT_PATH,
            argv=fixed_scene_argv,
            eval_episodes=EVALUATION_EPISODES,
            eval_max_steps=EVALUATION_MAX_STEPS,
            progress_callback=record_fixed_evaluation_progress,
            progress_step_interval=EVALUATION_PROGRESS_INTERVAL,
            fixed_hub_positions=FIXED_RENDER_HUB_POSITIONS,
            fixed_food_positions=FIXED_RENDER_FOOD_POSITIONS,
        )
    finally:
        fixed_evaluation_bar.close()
    fixed_evaluation_metrics = {
        key.replace("eval_", "eval_fixed_", 1): value
        for key, value in raw_fixed_evaluation_metrics.items()
    }
    fixed_baseline = fixed_evaluation_metrics["eval_fixed_frozen_vs_frozen_mean_delivery_difference"]
    fixed_evaluation_metrics["eval_fixed_learner_team0_advantage_vs_frozen_baseline"] = (
        fixed_evaluation_metrics["eval_fixed_learner_vs_frozen_mean_delivery_difference"]
        - fixed_baseline
    )
    fixed_evaluation_metrics["eval_fixed_learner_team1_advantage_vs_frozen_baseline"] = (
        fixed_evaluation_metrics["eval_fixed_frozen_vs_learner_mean_delivery_difference"]
        + fixed_baseline
    )
    fixed_evaluation_metrics.update({
        "checkpoint": str(ACTIVE_CHECKPOINT_PATH),
        "fixed_hub_positions": FIXED_RENDER_HUB_POSITIONS,
        "fixed_food_positions": FIXED_RENDER_FOOD_POSITIONS,
        "fixed_food_count": FIXED_RENDER_FOOD_COUNT,
        "fixed_food_sources": len(FIXED_RENDER_FOOD_POSITIONS),
        "eval_episodes": EVALUATION_EPISODES,
        "eval_max_steps": EVALUATION_MAX_STEPS,
    })
    fixed_evaluation_artifact = CURRICULUM_ROOT / f"fixed_center_eval_{EVALUATION_EPISODES}ep_{EVALUATION_MAX_STEPS}step.json"
    fixed_evaluation_artifact.write_text(
        json.dumps(fixed_evaluation_metrics, indent=2, sort_keys=True),
        encoding="utf-8",
    )
else:
    fixed_evaluation_progress_rows = []
    fixed_evaluation_metrics = {"status": "missing_checkpoint", "checkpoint": str(ACTIVE_CHECKPOINT_PATH) if ACTIVE_CHECKPOINT_PATH else None}
    fixed_evaluation_artifact = None

{
    "fixed_evaluation_artifact": str(fixed_evaluation_artifact) if fixed_evaluation_artifact else None,
    "fixed_evaluation_metrics": fixed_evaluation_metrics,
    "fixed_evaluation_progress_rows": fixed_evaluation_progress_rows,
}


## Render

In [ ]:
if ACTIVE_CHECKPOINT_PATH is not None and ACTIVE_CHECKPOINT_PATH.exists():
    rollout_video = render_adversarial_rollout(
        ACTIVE_CHECKPOINT_PATH,
        FIXED_RENDER_VIDEO_PATH,
        argv=fixed_scene_argv,
        max_frames=RENDER_MAX_FRAMES,
        tile_size=22,
        action_mode=spec.args["eval_action_mode"],
        fixed_hub_positions=FIXED_RENDER_HUB_POSITIONS,
        fixed_food_positions=FIXED_RENDER_FOOD_POSITIONS,
    )
    display(Video(str(rollout_video), embed=True))
else:
    rollout_video = None
    print(f"No checkpoint yet: {CHECKPOINT_PATH}; active fallback: {ACTIVE_CHECKPOINT_PATH}")

{
    "rollout_video": str(rollout_video) if rollout_video else None,
    "fixed_render_hub_positions": FIXED_RENDER_HUB_POSITIONS,
    "fixed_render_food_positions": FIXED_RENDER_FOOD_POSITIONS,
}
